In [1]:
# 强制 Keras 使用 JAX 作为底层计算后端
import os
os.environ["KERAS_BACKEND"] = "jax"

import jax
import jax.numpy as jnp
import keras
import numpy as np
import time

print(f"[System] JAX Computational Devices: {jax.devices()}")
print(f"[System] Keras Execution Backend: {keras.backend.backend()}")

def compute_bit_reversal(n):
    """计算长度为 n 的位反转排列 (n 必须是 2 的幂)"""
    k = int(np.log2(n))
    assert n == 2**k, "n 必须是 2 的整数次幂"
    bin_strs = [f"{i:0{k}b}" for i in range(n)]
    rev_bin_strs = [s[::-1] for s in bin_strs]
    perm = np.array([int(s, 2) for s in rev_bin_strs], dtype=np.int32)
    return perm

def naive_butterfly(x, weights, reversed_order=False):
    """
    纯 JAX 实现的蝶式矩阵乘法
    x: (Batch, N)
    weights: (k, N//2, 2, 2)
    """
    B, N = x.shape
    k = weights.shape[0]
    
    stages = range(k)
    if reversed_order:
        stages = reversed(stages)
        
    for s in stages:
        out_blocks = N // (2**(s+1))
        in_block = 2**s
        
        x_reshaped = x.reshape((B, out_blocks, 2, in_block))
        W_s = weights[s].reshape((out_blocks, in_block, 2, 2))
        
        x_reshaped = jnp.einsum('b o c i, o i d c -> b o d i', x_reshaped, W_s)
        x = x_reshaped.reshape((B, N))
        
    return x

class ButterflyLinear(keras.layers.Layer):
    """Keras 3 兼容的蝶式全连接层"""
    def __init__(self, n, n_stacks=1, **kwargs):
        super().__init__(**kwargs)
        self.n = n
        self.k = int(np.log2(n))
        self.n_stacks = n_stacks
        assert n == 2**self.k, "n 必须是 2 的整数次幂"

    def build(self, input_shape):
        init = keras.initializers.RandomNormal(stddev=1.0 / np.sqrt(2))
        
        self.weights_fwd = self.add_weight(
            shape=(self.k, self.n // 2, 2, 2),
            initializer=init,
            trainable=True,
            name="weights_fwd"
        )
        
        if self.n_stacks == 2:
            self.weights_rev = self.add_weight(
                shape=(self.k, self.n // 2, 2, 2),
                initializer=init,
                trainable=True,
                name="weights_rev"
            )
            
        self.perm = jnp.array(compute_bit_reversal(self.n))

    def call(self, x):
        x = x[:, self.perm]
        
        x = naive_butterfly(x, self.weights_fwd, reversed_order=False)
        x = x / jnp.sqrt(2.0)
        
        if self.n_stacks == 2:
            x = naive_butterfly(x, self.weights_rev, reversed_order=True)
            x = x / jnp.sqrt(2.0)
            
        return x

def build_model(model_type="dense"):
    inputs = keras.Input(shape=(28, 28))
    x = keras.layers.Flatten()(inputs)
    
    pad_width = 1024 - 784
    x = keras.ops.pad(x, [[0, 0], [0, pad_width]])
    
    if model_type == "dense":
        x = keras.layers.Dense(1024, activation="relu")(x)
        x = keras.layers.Dense(1024, activation="relu")(x)
    elif model_type == "b1":
        x = ButterflyLinear(1024, n_stacks=1)(x)
        x = keras.layers.Activation("relu")(x)
        x = ButterflyLinear(1024, n_stacks=1)(x)
        x = keras.layers.Activation("relu")(x)
    elif model_type == "b2":
        x = ButterflyLinear(1024, n_stacks=2)(x)
        x = keras.layers.Activation("relu")(x)
        x = ButterflyLinear(1024, n_stacks=2)(x)
        x = keras.layers.Activation("relu")(x)

    outputs = keras.layers.Dense(10, activation="softmax")(x)
    
    model = keras.Model(inputs, outputs, name=f"Fashion_MNIST_{model_type}")
    return model

# 1. 加载 Fashion-MNIST 数据
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train, x_test = x_train.astype("float32") / 255.0, x_test.astype("float32") / 255.0

# 2. 编译并打印参数量对比
model_dense = build_model("dense")
model_b2 = build_model("b2")

print("\n-------------------------------------------------------")
print("[Model Topology] Parameter Capacity Analysis")
print("-------------------------------------------------------")
print(f"  Configuration [DENSE] Total Parameters: {model_dense.count_params():,}")
print(f"  Configuration [B2]    Total Parameters: {model_b2.count_params():,}")
print("-------------------------------------------------------\n")

optimizer = keras.optimizers.AdamW(learning_rate=1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy()

model_dense.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])
model_b2.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])

# 3. 训练 B2 蝶式模型
print("[Training] Commencing optimization sequence for B2 configuration. Epochs: 5")
history_b2 = model_b2.fit(
    x_train, y_train, 
    batch_size=128, 
    epochs=5, 
    validation_data=(x_test, y_test)
)

# ==========================================
# Phase 2 
# ==========================================
import time
import jax
import jax.numpy as jnp

N = 8192       
BATCH = 1024   
K = int(np.log2(N)) 

print(f"\n[Benchmark] Initializing tensor memory allocation. Shape Constraints: Batch={BATCH}, N={N}, K={K}")

dtype = jnp.float32 
key = jax.random.PRNGKey(42)
k1, k2, k3, k4 = jax.random.split(key, 4)

x_dummy = jax.random.normal(k1, (BATCH, N), dtype=dtype)
dense_w = jax.random.normal(k2, (N, N), dtype=dtype)

bfly_w_fwd = jax.random.normal(k3, (K, N//2, 2, 2), dtype=dtype)
bfly_w_rev = jax.random.normal(k4, (K, N//2, 2, 2), dtype=dtype)
perm = jnp.array(compute_bit_reversal(N))

@jax.jit
def run_dense(x, w):
    return x @ w

@jax.jit
def run_butterfly(x, w_fwd, w_rev, p):
    x = x[:, p]
    x = naive_butterfly(x, w_fwd, reversed_order=False)
    x = naive_butterfly(x, w_rev, reversed_order=True)
    return x

# Warmup
print("[JIT] Executing compiler Ahead-of-Time optimizations...")
print("  -> Compiling [Dense] kernel...")
_ = run_dense(x_dummy, dense_w).block_until_ready()

print("  -> Compiling [Butterfly] kernel...")
_ = run_butterfly(x_dummy, bfly_w_fwd, bfly_w_rev, perm).block_until_ready()

# 测速
iterations = 50

print(f"[Benchmark] Evaluating execution latency (Iterations={iterations})...")
start = time.perf_counter()
for _ in range(iterations):
    out_dense = run_dense(x_dummy, dense_w)
out_dense.block_until_ready()  
dense_time = (time.perf_counter() - start) / iterations * 1000  

start = time.perf_counter()
for _ in range(iterations):
    out_bfly = run_butterfly(x_dummy, bfly_w_fwd, bfly_w_rev, perm)
out_bfly.block_until_ready()
bfly_time = (time.perf_counter() - start) / iterations * 1000

print("\n=======================================================")
print(f"  Micro-Operator Latency Benchmark Result")
print(f"  Shape Definitions: Batch={BATCH}, N={N}")
print("=======================================================")
print(f"  Dense Linear (Baseline)       : {dense_time:.4f} ms")
print(f"  Butterfly Linear (Structured) : {bfly_time:.4f} ms")
print("-------------------------------------------------------")
print(f"  Latent Ratio (Butterfly / Dense) = {bfly_time / dense_time:.2f}x")
print("=======================================================\n")

[System] JAX Computational Devices: [CudaDevice(id=0)]
[System] Keras Execution Backend: jax
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

-------------------------------------------------------
[Model Topology] Parameter Capacity Analysis
-------------------------------------------------------
  Configuration [DENSE] Total Parameters: 2,109,450
  Configuration [B2]    Total Parameters: 92,170
-------------------------------------------------------

[Training] Commencing optimization sequence for B2 configuration. Epochs: 5
Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.6128 - loss: 1.0922 - val_accuracy: 0.8093 - val_loss: 0.5215
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8310 - loss: 0.4689 - val_accuracy: 0.8310 - val_loss: 0.4641
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8524 -